In [2]:
import json

In [3]:
# Reading a single metadata or pdf_parses file
file_path = "/media/parthosarothi/Studio/UbuntuWorkSpace/SemanticScholarDataset/20260313_115134_00043_vcx4r_0a24153e-8434-4220-9c25-922408f36db8.jsonl"

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        paper_data = json.loads(line)
        # Access data fields like 'title', 'abstract', or 'mag_id'
        # print(paper_data.get('title'))
        break


In [4]:
paper_data.keys()

dict_keys(['corpusid', 'externalids', 'content'])

In [5]:
paper_data["corpusid"]

10412377

In [6]:
raw_content = paper_data["content"]
raw_content

{'source': {'pdfurls': None,
  'pdfsha': '60830c951d1fb5f6b2a3b1d1a651fdf22a918657',
  'oainfo': {'license': 'CCBY',
   'openaccessurl': 'https://link.springer.com/content/pdf/10.1186/s40201-015-0160-8.pdf',
   'status': 'HYBRID'}},
 'text': "\nSuitability of electrolyzed oxidizing water for the disinfection of hard surfaces and equipment in radiology\n\n\nRobert Pintaric \nJoze Matela \nStefan Pintaric \nSuitability of electrolyzed oxidizing water for the disinfection of hard surfaces and equipment in radiology\n10.1186/s40201-015-0160-8R E S E A R C H A R T I C L E Open Access\nBackground: Hospitals are faced with increasingly resistant strains of micro-organisms. When it comes to disinfection, individual parts of electronic equipment of angiology diagnostics such as patient couches of computer tomography (CT) and magnetic resonance imaging (MRI) scanners prove to be very hard to disinfect. Disinfectants of choice are therefore expected to possess properties such as rapid, residue-fr

In [7]:
def extract_info(raw_text:str,annotations:dict,info_type:str):
    annotations = eval(annotations[info_type]) # this can be more than one entries
    attribute_texts = []
    print(f'Total {len(annotations)} markers')
    for item in annotations:
        start = int(item['start'])
        end = int(item["end"])
        data = raw_text[start:end]
        attribute_texts.append(data)
    return attribute_texts

def extract_info_w_attributes(raw_text:str,annotations:dict,info_type:str):
    annotations = eval(annotations[info_type]) # this can be more than one entries
    attribute_texts = []
    print(f'Total {len(annotations)} markers')
    for item in annotations: # here we can have an attribute key as well
        start = int(item['start'])
        end = int(item["end"])
        if 'attributes' in item.keys():
            attribute = item['attributes']
            attribute_vals = []
            for k in attribute.keys():
                at_val = attribute[k]
                attribute_vals.append(at_val)
        else:
            attribute_vals = None
        data = raw_text[start:end]
        attribute_texts.append((attribute_vals,data))
    return attribute_texts

In [10]:
all_text = raw_content["text"]
all_annotations = raw_content["annotations"]
info = extract_info_w_attributes(all_text,all_annotations,"bibentry")
for i in info:
    print(i)

Total 39 markers
(['b0'], 'Practical aspects of personal exposure monitoring in hospitals and research laboratories. Aubert B , Radioprotection. 26Aubert B. Practical aspects of personal exposure monitoring in hospitals and research laboratories. Radioprotection. 1991;26:481-91.')
(['b1', 36244258], 'Potential exposure to hepatitis C virus through accidental blood contact in interventional radiology. N Baffoy-Fayard, S Maugat, M Sapoval, P Cluzel, A Denys, N Sellier, J Vasc Interv Radiol. 14Baffoy-Fayard N, Maugat S, Sapoval M, Cluzel P, Denys A, Sellier N, et al. Potential exposure to hepatitis C virus through accidental blood contact in interventional radiology. J Vasc Interv Radiol. 2003;14:173-9.')
(['b2', 28664063], 'Environmental radioactivity, population exposure and related health risks in the east Baltic region. M I Balonov, E M Krisyuk, C Ramel, Scand J Work Environ Health. 25Balonov MI, Krisyuk EM, Ramel C. Environmental radioactivity, population exposure and related health 

In [11]:
# general attributes
all_text = raw_content["text"]
attribute = 'title'
annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
all_markers = []
for item in annotation:
    start = int(item['start'])
    end = int(item["end"])
    all_markers.append((start,end))
attribute_text = "" # empty string
for marker in all_markers: # each element is (start, end)
    attribute_text += all_text[marker[0]:marker[1]] + "\n"
attribute_text

'Suitability of electrolyzed oxidizing water for the disinfection of hard surfaces and equipment in radiology\nSuitability of electrolyzed oxidizing water for the disinfection of hard surfaces and equipment in radiology\n'

In [6]:
def get_regular_attributes(raw_content:dict,attribute:str):
    # general attributes
    all_text = raw_content["text"]
    annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
    all_markers = []
    for item in annotation:
        start = int(item['start'])
        end = int(item["end"])
        all_markers.append((start,end))
    attribute_text = "" # empty string
    for marker in all_markers: # each element is (start, end)
        attribute_text += all_text[marker[0]:marker[1]] + "\n"
    return attribute_text

def get_citations(raw_content:dict):
    # only for bibentry
    all_text = raw_content["text"]
    attribute = 'bibentry'
    annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
    all_markers = []
    for item in annotation:
        cite_id = item["attributes"]["id"]
        if "matched_paper_id" in item["attributes"].keys():
            cite_paper_id = item["attributes"]["matched_paper_id"]
        else:
            cite_paper_id = -1
        start = int(item['start'])
        end = int(item["end"])
        all_markers.append((start,end,cite_id,cite_paper_id))
    attribute_text = []
    for marker in all_markers: # each element is (start, end)
        data = {"id":marker[2],"paper":all_text[marker[0]:marker[1]],"ss_id":marker[-1]}
        attribute_text.append(data)
    return attribute_text

In [8]:
attribute_text = get_regular_attributes(raw_content,"abstract")
print(attribute_text)

Background: Hospitals are faced with increasingly resistant strains of micro-organisms. When it comes to disinfection, individual parts of electronic equipment of angiology diagnostics such as patient couches of computer tomography (CT) and magnetic resonance imaging (MRI) scanners prove to be very hard to disinfect. Disinfectants of choice are therefore expected to possess properties such as rapid, residue-free action without any damaging effect on the sensitive electronic equipment. This paper discusses the use of the neutral electrolyzed oxidizing water (EOW) as a biocide for the disinfection of diagnostic rooms and equipment. Methods: The CT and MRI rooms were aerosolized with EOW using aerosolization device. The presence of micro-organisms before and after the aerosolization was recorded with the help of sedimentation and cyclone air sampling. Total body count (TBC) was evaluated in absolute and log values.Results: The number of micro-organisms in hospital rooms was low as expecte

In [45]:
# only for bibentry
all_text = raw_content["text"]
attribute = 'bibentry'
annotation = eval(raw_content["annotations"][attribute]) # this can be more than one entries
all_markers = []
for item in annotation:
    cite_id = item["attributes"]["id"]
    if "matched_paper_id" in item["attributes"].keys():
        cite_paper_id = item["attributes"]["matched_paper_id"]
    else:
        cite_paper_id = -1
    start = int(item['start'])
    end = int(item["end"])
    all_markers.append((start,end,cite_id,cite_paper_id))
attribute_text = []
for marker in all_markers: # each element is (start, end)
    data = {"id":marker[2],"paper":all_text[marker[0]:marker[1]],"ss_id":marker[-1]}
    attribute_text.append(data)
for d in attribute_text:
    print(f"[{d["id"]}]\nSS ID:{d["ss_id"]}\n{d["paper"]}\n-------")

[b0]
SS ID:-1
Practical aspects of personal exposure monitoring in hospitals and research laboratories. Aubert B , Radioprotection. 26Aubert B. Practical aspects of personal exposure monitoring in hospitals and research laboratories. Radioprotection. 1991;26:481-91.
-------
[b1]
SS ID:36244258
Potential exposure to hepatitis C virus through accidental blood contact in interventional radiology. N Baffoy-Fayard, S Maugat, M Sapoval, P Cluzel, A Denys, N Sellier, J Vasc Interv Radiol. 14Baffoy-Fayard N, Maugat S, Sapoval M, Cluzel P, Denys A, Sellier N, et al. Potential exposure to hepatitis C virus through accidental blood contact in interventional radiology. J Vasc Interv Radiol. 2003;14:173-9.
-------
[b2]
SS ID:28664063
Environmental radioactivity, population exposure and related health risks in the east Baltic region. M I Balonov, E M Krisyuk, C Ramel, Scand J Work Environ Health. 25Balonov MI, Krisyuk EM, Ramel C. Environmental radioactivity, population exposure and related health r

'bibref' - ids for citations